# Réplica del Fuzzy Logic Controller for a Traffic Junction (Pappis & Mamdani, 1977)

Abre este notebook en VS Code (`Ctrl+Shift+P` -> `Notebook: Open Notebook`) con el kernel del entorno `.venv`.
Ejecutá celda a celda con `Ctrl+Shift+Enter`.

In [ ]:
import sys; sys.path.insert(0, "../codigo")  # el codigo vive en codigo/
import numpy as np
import fuzzy_logic as fl
import traffic_model as tm
from traffic_model import (simulate_fixed_cycle, simulate_fuzzy,
                           webster_optimum, COMBINATIONS, run_table, LITERAL, CALIBRADA)

## 1) Verificación del controlador difuso
Reproduce el ejemplo trabajado del paper (pág. 7): debe seleccionar una extensión de 10 s.

In [ ]:
fl._selftest()

## 2) Una sola réplica (para inspeccionar)
Retardo medio por vehículo (s) para un flujo asimétrico.

In [ ]:
ns, ew = 360, 2520          # veh/h
lam_ns, lam_ew = ns/3600, ew/3600
dm_fixed = simulate_fixed_cycle(lam_ns, lam_ew, seed=1, duration=7200)
dm_fuzzy = simulate_fuzzy(lam_ns, lam_ew, seed=1, duration=7200, **CALIBRADA)
print(f'ciclo fijo (veh-act): {dm_fixed:6.2f} s/veh')
print(f'controlador difuso  : {dm_fuzzy:6.2f} s/veh')

## 3) Tabla VIII completa (promedia varias réplicas)
Esto tarda ~1 min. Columnas: flujos, retardo ciclo fijo (línea base), difuso y % de mejora.

In [ ]:
rows = run_table(reps=20, duration=7200, base_seed=0, **CALIBRADA)
print(f"{'NS':>5} {'EW':>5} | {'fijo':>6} {'fuzzy':>6} {'mej%':>7}")
print('-'*34)
for (n,e,dmf,dmz,imp) in rows:
    print(f'{n:5d} {e:5d} | {dmf:6.2f} {dmz:6.2f} {imp:6.1f}%')

## 4) Las dos lecturas del controlador
El paper fija reglas y conjuntos, pero no dice con precisión cómo su programa obtenía A y Q.
Ver `README.md` (sección *Interpretaciones*) y `fidelity_audit.py`.
- `LITERAL`: A = arribos acumulados, Q = cola total del brazo en rojo, evaluada en la cola predicha β′ (ejemplo de la pág. 7).
- `CALIBRADA`: A = vehículos que cruzan (limitados por saturación), Q = llegados al brazo en rojo desde el inicio del verde.

In [ ]:
for name, cfg in [('literal', LITERAL), ('calibrada', CALIBRADA)]:
    d = simulate_fuzzy(360/3600, 2520/3600, seed=1, duration=7200, **cfg)
    print(f'{name:9s} -> dm={d:6.2f} s/veh')